A simple script to build Casmopolitan optimizer and run it on Ackley53D.

In [ ]:
import torch
import numpy as np
from mcbo import task_factory
from mcbo.optimizers.bo_builder import BoBuilder

if __name__ == '__main__':

    # Building Ackley 53D
    num_dims = [50, 3]
    lower_bounds = np.array([0.0] * num_dims[0] + [-1.0] * num_dims[1])
    upper_bounds = np.array([1.0] * num_dims[0]+ [1.0] * num_dims[1])
    task_kws = {'variable_type': ['nominal', 'num'], 
                'num_dims': num_dims,
                'num_categories': [2, 0],
                'lb': lower_bounds,
                'ub': upper_bounds}
    task = task_factory(task_name='ackley', **task_kws)
    search_space = task.get_search_space()

    # Building GP like in Casmopolitan
    bo_builder = BoBuilder(
        model_id='gp_to', 
        acq_opt_id='is', 
        acq_func_id='ei', 
        tr_id='basic', 
        init_sampling_strategy="sobol_scramble"
    )

    optimizer = bo_builder.build_bo(search_space=search_space, n_init=20, device=torch.device("cuda"))

    for i in range(100):
        x = optimizer.suggest(1)
        y = task(x)
        optimizer.observe(x, y)
        print(f'Iteration {i + 1:3d}/{100:3d} - f(x) = {y[0, 0]:.3f} - f(x*) = {optimizer.best_y.item():.3f}')

    # Access history of suggested points and black-box values
    all_x = search_space.inverse_transform(optimizer.data_buffer.x)
    all_y = optimizer.data_buffer.y.cpu().numpy()